# 1. Pre-Procesamiento Inicial

### Carga de datos

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import ast

ruta='../data/listings.csv'
df=pd.read_csv(ruta)

df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,96033,https://www.airbnb.com/rooms/96033,20250930030808,2025-09-30,city scrape,"Bonito piso a 200m de la playa, El Palo (Málaga)",Do you have a backpacker spirit and are lookin...,"200 metres from the beaches of El Palo, Malaga...",https://a0.muscache.com/pictures/hosting/Hosti...,510467,...,4.93,4.44,4.61,ESFCTU0000290200003588210000000000000000VUT/MA...,f,1,1,0,0,1.88
1,166473,https://www.airbnb.com/rooms/166473,20250930030808,2025-09-30,city scrape,Perfect Location In Malaga,This apartment is rented out by the room - new...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,793360,...,4.91,4.80,4.72,NaN,f,5,1,4,0,0.59
2,330760,https://www.airbnb.com/rooms/330760,20250930030808,2025-09-30,city scrape,Malaga Lodge Guesthouse Double room-shared bath.,The Lodge is set in a charming town house in L...,Málaga Lodge is situated next to the famous Sa...,https://a0.muscache.com/pictures/85419390/38a9...,1687526,...,4.62,4.52,4.48,ESHFTU0000290200004234200060000000000000VFT/MA...,t,6,4,2,0,0.41
3,340024,https://www.airbnb.com/rooms/340024,20250930030808,2025-09-30,city scrape,NEW APARTMENT IN MALAGA CENTER,Welcome to Málaga!<br />This is a modern and e...,It is a central area and has all kinds of serv...,https://a0.muscache.com/pictures/hosting/Hosti...,1725690,...,4.79,4.72,4.77,VFT/MA/02334,f,1,1,0,0,2.11
4,358541,https://www.airbnb.com/rooms/358541,20250930030808,2025-09-30,city scrape,Casa La Maga - Apartment for happy people,"For years, Raúl and I were super happy in this...",The apartment is in the very heart of Malaga C...,https://a0.muscache.com/pictures/miso/Hosting-...,1526932,...,4.97,4.80,4.78,VFT/MA/02288,f,1,1,0,0,2.48


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9714 entries, 0 to 9713
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            9714 non-null   int64  
 1   listing_url                                   9714 non-null   object 
 2   scrape_id                                     9714 non-null   int64  
 3   last_scraped                                  9714 non-null   object 
 4   source                                        9714 non-null   object 
 5   name                                          9714 non-null   object 
 6   description                                   9477 non-null   object 
 7   neighborhood_overview                         4170 non-null   object 
 8   picture_url                                   9714 non-null   object 
 9   host_id                                       9714 non-null   i

### Nulos por columnas

In [3]:
pd.set_option('display.max_rows', None)  # Mostrar todas las filas
df.isnull().sum().sort_values(ascending=False)

neighbourhood_group_cleansed                    9714
calendar_updated                                9714
host_neighbourhood                              8468
neighbourhood                                   5544
neighborhood_overview                           5544
host_about                                      4174
host_location                                   2280
review_scores_accuracy                          1005
first_review                                    1005
last_review                                     1005
review_scores_cleanliness                       1005
review_scores_location                          1005
review_scores_rating                            1005
reviews_per_month                               1005
review_scores_value                             1005
review_scores_communication                     1005
review_scores_checkin                           1005
host_response_rate                               929
host_response_time                            

In [4]:
# Columnas a eliminar por tener +4000 nulos, que sería aprox o más de la mitad del dataset
cols_to_drop = [
    'neighbourhood_group_cleansed',
    'calendar_updated',
    'host_neighbourhood',
    'neighbourhood',
    'neighborhood_overview',
    'host_about'
]

df = df.drop(columns=cols_to_drop)
print(f"Columnas restantes: {df.shape[1]}")

Columnas restantes: 73


### Limpieza de variables

In [5]:
# ── 2. LIMPIEZA DE PRECIO ───────────────────────────────────────────
df['price'] = (
    df['price']
    .str.replace('[$,]', '', regex=True)
    .astype(float)
)

# ── 3. LIMPIEZA DE BAÑOS ─────────────────────────────────────────────────────
# Extraemos la característica baño privado
# 4a. Variable binaria ANTES de perder la info textual
#     1 = baño privado, 0 = baño compartido
df['private_bathroom'] = (~df['bathrooms_text']
                            .str.lower()
                            .str.contains('shared', na=False)
                        ).astype(int)

# 4b. Ahora sí extraemos el número de baños del alojamiento
df['bathrooms'] = df['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)
df.drop(columns=['bathrooms_text'], inplace=True)

# ── 4. ONE-HOT ENCODING DE AMENITIES - PALABRAS CLAVE POR TUPLAS (Top 22) ───────────────────────────────
TOP_AMENITIES = {
    'has_kitchen':            ('Kitchen',),
    'has_wifi':               ('Wifi',),
    'has_hair_dryer':         ('Hair dryer',),
    'has_iron':               ('Iron',),
    'has_hot_water':          ('Hot water',),
    'has_bed_linens':         ('Bed linens',),
    'has_microwave':          ('Microwave',),
    'has_hangers':            ('Hangers',),
    'has_refrigerator':       ('Refrigerator',),
    'has_essentials':         ('Essentials',),
    'has_cooking_basics':     ('Cooking basics',),
    'has_tv':                 ('TV',),
    'has_air_conditioning':   ('Air conditioning','AC'),
    'has_washer':             ('Washer',),
    'has_shampoo':            ('Shampoo',),
    'has_toaster':            ('Toaster',),
    'has_heating':            ('Heating',),
    'has_freezer':            ('Freezer',),
    'has_coffee_maker':       ('Coffee maker',),
    'has_dishes':             ('Dishes and silverware'),
    'has_balcony_or_terrace': ('Balcony', 'Terrace'),  # ← dos keywords
}

def parse_amenities(text):
    try:
        return ast.literal_eval(text)
    except:
        return []

amenity_lists = df['amenities'].apply(parse_amenities)
#Bucle adapatado a tuplas
for col_name, keywords in TOP_AMENITIES.items():
    df[col_name] = amenity_lists.apply(
        lambda lst, kws=keywords: 1 if any(
            kw.lower() in item.lower() for item in lst for kw in kws
        ) else 0
    )

# Verificación rápida de alojamientos con cada amenitie
print(df[[c for c in TOP_AMENITIES.keys()]].sum().sort_values())

# ── 5. GUARDADO CSV V1 ───────────────────────────────────────────────────
df.to_csv('../data/listingV1.csv', index=False)
print("✅ Dataset guardado en ../data/listingV1.csv")

has_balcony_or_terrace    3655
has_freezer               5606
has_toaster               5742
has_shampoo               6114
has_cooking_basics        7040
has_essentials            7275
has_hangers               7356
has_microwave             7489
has_bed_linens            7505
has_heating               7606
has_coffee_maker          7657
has_refrigerator          7702
has_iron                  7851
has_hair_dryer            8121
has_hot_water             8205
has_washer                8613
has_tv                    9097
has_kitchen               9278
has_air_conditioning      9419
has_wifi                  9498
has_dishes                9710
dtype: int64
✅ Dataset guardado en ../data/listingV1.csv


### Transformar porcentajes en valor numérico

In [6]:
for col in ['host_acceptance_rate', 'host_response_rate']:
    df[col] = df[col].str.replace('%', '').astype(float) / 100

print(df[['host_acceptance_rate', 'host_response_rate']].describe())


       host_acceptance_rate  host_response_rate
count           9071.000000         8785.000000
mean               0.931912            0.960839
std                0.183753            0.129283
min                0.000000            0.000000
25%                0.980000            0.990000
50%                1.000000            1.000000
75%                1.000000            1.000000
max                1.000000            1.000000


### Transformar columnas binarias de texto 't' o 'f' a numéricas

In [7]:
cols_binarias = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'has_availability',
    'instant_bookable'
]

df[cols_binarias] = df[cols_binarias].replace({'t': 1, 'f': 0})


C:\Users\david\AppData\Local\Temp\ipykernel_25988\3262820833.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols_binarias] = df[cols_binarias].replace({'t': 1, 'f': 0})


### Eliminar AirBnBs con precios absurdos/pisos fantasma

In [8]:
# Por ejemplo, menores a 10€ noche o mayores de 5000€
df = df[(df['price'] >= 10) & (df['price'] <= 5000)]

# Eliminar pisos fantasma (que tengan menos de 3 reseñas)
df = df[df['number_of_reviews'] >= 3]  # mínimo 3 reseñas para considerar el piso activo

df.to_csv('../data/listingV2.csv', index=False)
print("✅ Dataset guardado en ../data/listingV2.csv")

✅ Dataset guardado en ../data/listingV2.csv


### Matriz de correlación con precio/noche

In [9]:
#Correlacion de las columnas con el precio por noche
import plotly.graph_objects as go

correlacion = df.corr(numeric_only=True)['price'].drop('price').sort_values()

colors = ['#ef4444' if v < 0 else '#3b82f6' for v in correlacion.values]

fig = go.Figure(go.Bar(
    x=correlacion.values,
    y=correlacion.index,
    orientation='h',
    marker_color=colors,
    text=[f"{v:.3f}" for v in correlacion.values],
    textposition='outside'
))

fig.update_layout(title="Correlación variables numéricas con Price", height=1400)
fig.update_xaxes(title_text="Correlación r")
fig.update_yaxes(title_text="Variable")
fig.add_vline(x=0.1, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=-0.1, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()

`scrape_id` y `has_availability` salen como NaN, lo que quiere decir que todos sus valores son iguales, por lo que van a ser candidatas a eliminar

In [10]:
from scipy import stats
import pandas as pd

cols_categoricas = df.select_dtypes(include='object').columns.tolist()

resultados_anova = []

for col in cols_categoricas:
    grupos = [df[df[col] == cat]['price'].dropna()
            for cat in df[col].dropna().unique()
            if len(df[df[col] == cat]['price'].dropna()) > 1]  # evitar grupos vacíos
    
    if len(grupos) > 1:  # ANOVA necesita al menos 2 grupos
        f_stat, p_valor = stats.f_oneway(*grupos)
        resultados_anova.append({'columna': col, 'p_valor': round(p_valor, 6)})
    else:
        resultados_anova.append({'columna': col, 'p_valor': None})

resultados_anova_df = pd.DataFrame(resultados_anova).sort_values('p_valor')
print(resultados_anova_df.to_string())



                   columna   p_valor
3                     name  0.000000
4              description  0.000000
5              picture_url  0.000000
6                 host_url  0.000000
7                host_name  0.000000
8               host_since  0.000000
12        host_picture_url  0.000000
11      host_thumbnail_url  0.000000
15           property_type  0.000000
16               room_type  0.000000
13      host_verifications  0.000000
14  neighbourhood_cleansed  0.000000
17               amenities  0.000000
21                 license  0.000000
9            host_location  0.000074
20             last_review  0.007653
10      host_response_time  0.166385
19            first_review  0.996220
0              listing_url       NaN
1             last_scraped       NaN
2                   source       NaN
18   calendar_last_scraped       NaN


Únicas variables categóricas con algo de influencia en el precio `first_review`, `last_review` y `host_response_time` y algo `host_location`. La lectura que podemos hacer es que obtener reviews es fundamental, cuanto antes mejor, y que el tiempo de respuesta del anfitrión y su localización pueden influir en el precio, suponiendo que es indicativo de una mejor atención y soporte al cliente.

In [11]:
# Variables a eliminar por escasa relevancia
cols_eliminar = [
    # Metadatos scraping
    'scrape_id', 'has_availability', 'calendar_last_scraped', 'source', 'last_scraped',
    # URLs sin valor
    'host_thumbnail_url', 'host_picture_url',
    # Sin valor predictivo
    'host_name', 'license', 'first_review', 'last_review', 'host_location',
    # Redundantes con minimum/maximum_nights
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'minimum_maximum_nights', 'maximum_maximum_nights',
    'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm'
]

df = df.drop(columns=cols_eliminar)
print(f"Columnas restantes: {df.shape[1]}")

df.to_csv('../data/listingV3.csv', index=False)
print("✅ Dataset guardado en ../data/listingV3.csv")



Columnas restantes: 76
✅ Dataset guardado en ../data/listingV3.csv


In [12]:
# Eliminar apartamentos sin imagen
# Debemos eliminar estos apartamentos ya que al cambiar su imagen de portada, 
# la URL del CSV se encontraba desactualizada y no pudo descargar su imagen,
# se ha decidido que deben eliminarse ya que no son demasiados y aún nos quedamos con un buen set de datos.

from pathlib import Path

# Recogemos todos los IDs que sí tienen imagen descargada en disco
img_dir = Path('../data/Front_Images/')
ids_con_imagen = {int(f.stem) for f in img_dir.iterdir() if f.is_file()}

# Filtramos el dataset: solo nos quedamos con filas que tienen imagen
filas_antes = len(df)
df = df[df['id'].isin(ids_con_imagen)]

print(f"Filas eliminadas: {filas_antes - len(df)}")
print(f"Dataset final: {df.shape}")

Filas eliminadas: 389
Dataset final: (6635, 76)


# 2. Ingeniería de características

En este preprocesamiento final nos quedaremos con los datos puramente valiosos para la red neuronal que analizará los datos técnicos. Por ello vamos a proceder a eliminar variables innecesarias y sin valor del dataset y a su vez, porteriormente, crearemos nuevas variables con información interesante para el estudio y entrenamiento de las redes neuronales.

### Eliminar apartamentos sin presunta actividad en el último año

Esto no se hizo antes ya que aunque un piso no haya estado activo el último año, sus fotos, junto a sus reseñas, aunque sean del pasado sirven para el entrenamiento de la el modelo neuronal convolucional de imágenes.

## 2.1. Eliminación de variables 

In [13]:
# Eliminar pisos sin actividad reciente
# Criterio: activo en el último año si cumple AL MENOS UNA de estas condiciones:
#   - Ocupado >= 30 días en los últimos 365 días (estimated_occupancy_l365d)
#   - >= 3 reseñas en los últimos 12 meses (number_of_reviews_ltm)
#   - >= 3 reseñas en el último año natural (number_of_reviews_ly)
# Usamos OR (|) porque basta con cumplir una para considerarlo activo

filas_antes = len(df)

mask_activo = (
    (df['estimated_occupancy_l365d'] >= 30) |
    (df['number_of_reviews_ltm']     >= 3)  |
    (df['number_of_reviews_ly']      >= 3)
)

df = df[mask_activo]

print(f"Pisos eliminados por inactividad: {filas_antes - len(df)}")
print(f"Dataset tras filtro: {df.shape}")

Pisos eliminados por inactividad: 839
Dataset tras filtro: (5796, 76)


Ahora volveremos a examinar como han quedado las correlaciones tras estos cambios

In [14]:
#Correlacion de las columnas con el precio por noche
import plotly.graph_objects as go

correlacion = df.corr(numeric_only=True)['price'].drop('price').sort_values()

colors = ['#ef4444' if v < 0 else '#3b82f6' for v in correlacion.values]

fig = go.Figure(go.Bar(
    x=correlacion.values,
    y=correlacion.index,
    orientation='h',
    marker_color=colors,
    text=[f"{v:.3f}" for v in correlacion.values],
    textposition='outside'
))

fig.update_layout(title="Correlación variables numéricas con Price", height=1400)
fig.update_xaxes(title_text="Correlación r")
fig.update_yaxes(title_text="Variable")
fig.add_vline(x=0.1, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=-0.1, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()

In [15]:
from scipy import stats
import pandas as pd

cols_categoricas = df.select_dtypes(include='object').columns.tolist()

resultados_anova = []

for col in cols_categoricas:
    grupos = [df[df[col] == cat]['price'].dropna()
            for cat in df[col].dropna().unique()
            if len(df[df[col] == cat]['price'].dropna()) > 1]  # evitar grupos vacíos
    
    if len(grupos) > 1:  # ANOVA necesita al menos 2 grupos
        f_stat, p_valor = stats.f_oneway(*grupos)
        resultados_anova.append({'columna': col, 'p_valor': round(p_valor, 6)})
    else:
        resultados_anova.append({'columna': col, 'p_valor': None})

resultados_anova_df = pd.DataFrame(resultados_anova).sort_values('p_valor')
print(resultados_anova_df.to_string())



                   columna  p_valor
1                     name  0.00000
2              description  0.00000
3              picture_url  0.00000
4                 host_url  0.00000
5               host_since  0.00000
7       host_verifications  0.00000
8   neighbourhood_cleansed  0.00000
9            property_type  0.00000
11               amenities  0.00000
10               room_type  0.00000
6       host_response_time  0.56752
0              listing_url      NaN


### Eliminación de columnas redundantes, innecesarias, inútiles o confusas para el entrenamiento

In [16]:
cols_eliminar = [

    'listing_url',       # URL del anuncio, solo identificador
    'host_url',          # URL del perfil del host
    'name',              # Título del anuncio — texto libre, cada nombre es único, no indica nada
    'description',       # Texto largo — NLP fuera del alcance del proyecto
    'amenities',         # Ya procesadas las mas importantes en columnas has_* → columna original redundante
    'picture_url',       # Ya se ha usado previamente, no sigue teniendo utilidad

# Nos quedamos con availability_365, lo demás es ruido
    'availability_30',
    'availability_60',
    'availability_90',
    'availability_eoy',   # redundante con availability_365

    'host_since',              # sin relevancia necesaria
    'host_verifications',      # cubierta por host_identity_verified
    'host_id',                 # identificador puro

    'number_of_reviews_l30d', # ventana demasiado corta, muy ruidosa
    'number_of_reviews_ly',   # redundante con ltm

    # solapadas por host_listings_count 
    'calculated_host_listings_count_shared_rooms', #subconjunto de columna existente 
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count',
    'host_total_listings_count',

    # correlaciones absurdas
    'has_dishes',
    'has_hangers',
    'has_hot_water',
    'host_acceptance_rate',
    'host_has_profile_pic',
    'has_essentials',
    'has_shampoo',
    'host_identity_verified',
    'has_wifi',
]
filas_antes = df.shape[1]
df = df.drop(columns=cols_eliminar)

print(f"Columnas eliminadas: {len(cols_eliminar)}")
print(f"Columnas restantes: {df.shape[1]}")

df.to_csv('../data/listingV4.csv', index=False)
print("✅ Dataset guardado en ../data/listingV4.csv")

Columnas eliminadas: 29
Columnas restantes: 47
✅ Dataset guardado en ../data/listingV4.csv


## 2.2. Creación de nuevas variables

In [17]:
import pandas as pd
import numpy as np

# Cargar tu dataset definitivo
df = pd.read_csv('../data/listingV4.csv')

# 1. Función Distancia al Centro de Málaga (Calle Larios aprox: 36.7212, -4.4213)
def haversine_distance(lat1, lon1, lat2=36.7212, lon2=-4.4213):
    R = 6371  # Radio de la Tierra en kilómetros
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

df['distancia_centro_km'] = haversine_distance(df['latitude'], df['longitude'])

# 2. Índice de Hacinamiento (Personas por habitación)
# Sumamos 1 a bedrooms para que los estudios (0 habs) no den error de división
df['personas_por_habitacion'] = df['accommodates'] / (df['bedrooms'] + 1)

# 3. Ratio de Comodidad (Baños por huésped)
df['banos_por_huesped'] = df['bathrooms'] / df['accommodates']

# 4. Amenities Score (Puntuación de servicios)
# Filtramos todas las columnas que creaste que empiezan por 'has_'
cols_amenities = [col for col in df.columns if col.startswith('has_')]
df['amenities_score'] = df[cols_amenities].sum(axis=1)

# 5. Precio por persona, ya que el precio se ve correlacionado en 0.52 con accomodates, 
# parece interesante ver esta variable
df['price_per_person'] = df['price'] / df['accommodates']

# 6. Distancia mínima a cualquier playa

PLAYAS_MALAGA = {
    # Zona Oeste (Límite Torremolinos hasta el Puerto)
    'El Cañuelo':      (36.6450, -4.4920),  # Límite con Torremolinos
    'Campo de Golf':   (36.6544, -4.4740),  # San Julián
    'Arraijanal':      (36.6600, -4.4680),  # Zona virgen
    'Guadalmar':       (36.6625, -4.4819),
    'Desembocadura':   (36.6690, -4.4590),  # Desembocadura del Guadalhorce
    'Sacaba':          (36.6855, -4.4578),
    'Misericordia':    (36.6961, -4.4447),
    'San Andres':      (36.7061, -4.4320),  # Huelin
    
    # Zona Centro (Este del Puerto)
    'Malagueta':       (36.7186, -4.4079),
    'La Caleta':       (36.7190, -4.3980),
    
    # Zona Este
    'Banos del Carmen':(36.7215, -4.3840),
    'Pedregalejo':     (36.7196, -4.3725),
    'Las Acacias':     (36.7190, -4.3650),  # Extensión de Pedregalejo
    'El Palo':         (36.7175, -4.3592),
    'El Candado':      (36.7145, -4.3467),
    'Penon del Cuervo':(36.7134, -4.3364),
    'El Cementerio':   (36.7130, -4.3320),  # Camino a La Araña
    'La Araña':        (36.7123, -4.3211),
    'El Hornillo':     (36.7100, -4.3150)   # Límite con Rincón de la Victoria
}

# Calculamos la distancia mínima a cualquier playa
df['distancia_playa_km'] = df.apply(
    lambda row: min(
        haversine_distance(row['latitude'], row['longitude'], lat, lon)
        for lat, lon in PLAYAS_MALAGA.values()
    ), axis=1
)

nuevas_variables = [
    'distancia_centro_km',
    'personas_por_habitacion',
    'banos_por_huesped',
    'amenities_score',
    'price_per_person',
    'distancia_playa_km'
]

df[nuevas_variables] = df[nuevas_variables].round(2)

# Comprobar las nuevas variables
print(df[['distancia_centro_km', 'personas_por_habitacion', 'banos_por_huesped', 'amenities_score', 'price_per_person', 'distancia_playa_km']].head())

# Sobrescribir o guardar como V5
df.to_csv('../data/listingV5.csv', index=False)

   distancia_centro_km  personas_por_habitacion  banos_por_huesped  \
0                 5.80                      1.0               0.33   
1                 5.37                      0.5               2.00   
2                 1.18                      1.0               0.75   
3                 1.00                      1.0               0.50   
4                 0.28                      1.0               0.50   

   amenities_score  price_per_person  distancia_playa_km  
0               13             19.33                0.41  
1               10             28.00                0.35  
2               14             30.00                1.34  
3               14             30.50                1.20  
4               13             43.50                1.32  
